In [8]:
# !pip install codecarbon
# !pip install -e .

import sys
sys.path.append("src")

import pandas as pd
import pickle
from trust_library.core import TrustEvaluator
from trust_library.factsheet import Factsheet
from trust_library.fairness import statistical_parity_difference

import matplotlib.pyplot as plt
import seaborn as sns
import json

from trust_library.utils import to_json_safe
import plotly.express as px

In [9]:
# factsheet = Factsheet.create_factsheet_interactive()
# factsheet = Factsheet(load_path="./factsheet.json")
# factsheet.save("./factsheet2.json")
factsheet = Factsheet(general={"target_column": "Target"}, fairness={"protected_feature": "Group","protected_values": [1],"favorable_outcomes": [1]}, save_path="./factsheet_actual.json")

with open("./models_and_data/model_tree.pkl", "rb") as f:
    loaded_model = pickle.load(f)

train_loaded = pd.read_csv("./models_and_data/train.csv")
test_loaded = pd.read_csv("./models_and_data/test.csv")

# USO DE MÉTRICA CONCRETA
y_pred = loaded_model.predict(
    test_loaded.drop(
        columns=[factsheet["general"]["target_column"]["value"]]
    )
)

group_mask = (
    test_loaded[factsheet["fairness"]["protected_feature"]["value"]]
    == factsheet["fairness"]["protected_values"]["value"][0]
)

# O bien
y_pred = loaded_model.predict(test_loaded.drop("Target", axis=1))
group_mask = test_loaded["Group"] == 1

statistical_parity_difference(y_pred,group_mask)

{'value': 0.37903622652710833,
 'favored_ratio_protected': 0.5891047998671317,
 'favored_ratio_unprotected': 0.2100685733400234,
 'n_protected': 6021,
 'n_unprotected': 5979,
 'n_protected_favored': 3547,
 'n_unprotected_favored': 1256}

In [10]:
evaluator = TrustEvaluator(
    model=loaded_model,
    train_data=train_loaded,
    test_data=test_loaded,
    factsheet=factsheet,
    config_path="./src/trust_library/configs.json"
)

subset = evaluator.evaluate(show_nan=True)
# Imprimimos subset formateado
print(json.dumps(subset, indent=4))

Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.02 seconds.
Metric 'class_imbalance' computed in 0.00 seconds.
Metric 'kl_divergence' computed in 0.00 se

In [11]:
import os
import urllib.request
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from typing import Protocol, runtime_checkable

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted

# Asumimos que tu clase TrustEvaluator está en core.py
# from core import TrustEvaluator 

# =====================================================================
# 1. EL CONTRATO (PROTOCOL)
# =====================================================================
# Esto garantiza que TrustEvaluator lance un error claro ANTES de hacer nada
# si el modelo no cumple con las interfaces de scikit-learn.

@runtime_checkable
class ScikitLearnModelContract(Protocol):
    def predict(self, X: pd.DataFrame | np.ndarray) -> np.ndarray: ...
    def predict_proba(self, X: pd.DataFrame | np.ndarray) -> np.ndarray: ...

def validate_model_contract(model):
    if not isinstance(model, ScikitLearnModelContract):
        raise TypeError(
            "El modelo proporcionado no cumple con el contrato ScikitLearnModelContract. "
            "Asegúrate de que el modelo implemente los métodos '.predict()' y '.predict_proba()'."
        )

# =====================================================================
# 2. DEFINICIÓN DEL MODELO PYTORCH (No muy pequeño)
# =====================================================================
class DeepTabularNet(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 2):
        super(DeepTabularNet, self).__init__()
        # Una red neuronal con múltiples capas ocultas
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# =====================================================================
# 3. EL WRAPPER (ADAPTER) DE PYTORCH A SCIKIT-LEARN
# =====================================================================
class PyTorchSklearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model: nn.Module, device: str = "cpu"):
        self.model = model
        self.device = torch.device(device)
        self.model.to(self.device)
        self.is_fitted_ = True # Asumimos que el modelo ya viene entrenado
        self.classes_ = np.array([0, 1]) # Asumimos clasificación binaria

    def predict_proba(self, X):
        self.model.eval()
        # Si X es un DataFrame, sacamos los valores
        if isinstance(X, pd.DataFrame):
            X = X.values
            
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            logits = self.model(X_tensor)
            probabilities = torch.softmax(logits, dim=1)
        return probabilities.cpu().numpy()

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

# =====================================================================
# 4. SCRIPT PRINCIPAL DE EJECUCIÓN
# =====================================================================
if __name__ == "__main__":
    print("-> 1. Descargando datasets mediante URL...")
    url_titanic = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    
    # Descargamos y preparamos un dataframe numérico limpio
    df = pd.read_csv(url_titanic)
    df = df[['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Survived']].dropna()
    
    # Split manual en Train y Test
    train_data = df.sample(frac=0.8, random_state=42)
    test_data = df.drop(train_data.index)

    # Preparar tensores por si necesitamos auto-entrenar
    features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
    X_train_tensor = torch.tensor(train_data[features].values, dtype=torch.float32)
    y_train_tensor = torch.tensor(train_data['Survived'].values, dtype=torch.long)

    print("-> 2. Instanciando la red neuronal profunda...")
    pytorch_model = DeepTabularNet(input_dim=len(features), num_classes=2)
    
    print("-> 3. Intentando descargar los pesos del modelo mediante URL...")
    # NOTA: Como no existe una URL pública estándar para un modelo tabular con esta arquitectura exacta,
    # simulamos la descarga. Si falla (que lo hará), hacemos un mini-entrenamiento de emergencia.
    model_url = "https://tu-repositorio-ficticio.com/mi_modelo_pesado.pth"
    model_path = "mi_modelo_pesado.pth"
    
    try:
        urllib.request.urlretrieve(model_url, model_path)
        pytorch_model.load_state_dict(torch.load(model_path))
        print("   ¡Modelo descargado y cargado con éxito!")
    except Exception as e:
        print("   [!] URL no accesible. Auto-entrenando el modelo durante 5 épocas para la demostración...")
        optimizer = torch.optim.Adam(pytorch_model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        pytorch_model.train()
        for epoch in range(5):
            optimizer.zero_grad()
            outputs = pytorch_model(X_train_tensor)
            loss = criterion(outputs, y_train_tensor)
            loss.backward()
            optimizer.step()
        pytorch_model.eval()
        print("   Auto-entrenamiento completado.")

    print("-> 4. Aplicando el Wrapper y validando el Contrato...")
    wrapped_model = PyTorchSklearnWrapper(model=pytorch_model)
    validate_model_contract(wrapped_model) # Si no cumple, explotará aquí

    print("-> 5. Preparando Factsheet y ejecutando TrustEvaluator...")

    factsheet = Factsheet(general={"target_column": "Survived"}, fairness={"protected_feature": "Pclass","protected_values": [3],"favorable_outcomes": [1]})

    # Asumiendo que TrustEvaluator está disponible en tu entorno
    evaluator = TrustEvaluator(
        model=wrapped_model,          # <- Pasamos el modelo envuelto
        train_data=train_data,
        test_data=test_data,
        factsheet=factsheet,
        config_path="./src/trust_library/configs.json"
    )
    
    subset = evaluator.evaluate(show_nan=True)
    print(json.dumps(subset, indent=4))

-> 1. Descargando datasets mediante URL...
-> 2. Instanciando la red neuronal profunda...
-> 3. Intentando descargar los pesos del modelo mediante URL...
   [!] URL no accesible. Auto-entrenando el modelo durante 5 épocas para la demostración...
   Auto-entrenamiento completado.
-> 4. Aplicando el Wrapper y validando el Contrato...
-> 5. Preparando Factsheet y ejecutando TrustEvaluator...
Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in

In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

# 1. URL DE DATOS REALES (Heart Disease Dataset)
url_data = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
df = pd.read_csv(url_data, names=columns, na_values="?")
df = df.dropna()

# Binarizar el target (0 = sin enfermedad, 1 = con enfermedad)
df["target"] = (df["target"] > 0).astype(int)

X = df.drop("target", axis=1).values
y = df["target"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. DEFINIR LA RED (Real y funcional)
class HeartDiseaseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )
    def forward(self, x):
        return self.net(x)

# 3. ENTRENAR Y GUARDAR (Simulando lo que harías en producción)
pytorch_model = HeartDiseaseNet(input_dim=X_train.shape[1])
optimizer = torch.optim.Adam(pytorch_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

for epoch in range(100): # Entrenamiento flash
    optimizer.zero_grad()
    loss = criterion(pytorch_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

# Guardamos el modelo "real"
torch.save(pytorch_model.state_dict(), "modelo_corazon_real.pth")
print("✅ Modelo guardado como 'modelo_corazon_real.pth'")

# ==========================================
# 4. USO DEL WRAPPER DEFINITIVO CON TU LIBRERÍA
# ==========================================
# Cargas el modelo desde el archivo
modelo_cargado = HeartDiseaseNet(input_dim=X_train.shape[1])
modelo_cargado.load_state_dict(torch.load("modelo_corazon_real.pth"))

# Aplicas el Wrapper Definitivo
wrapped_model = PyTorchSklearnWrapper(model=modelo_cargado)

print("✅ Wrapper aplicado. ¿Pasa partial_dependence? ->", hasattr(wrapped_model, "fit") and hasattr(wrapped_model, "predict_proba"))

✅ Modelo guardado como 'modelo_corazon_real.pth'
✅ Wrapper aplicado. ¿Pasa partial_dependence? -> False


In [13]:
from sklearn.model_selection import train_test_split

# 1. Hacemos el split manteniendo los DataFrames completos de Pandas
# (TrustEvaluator los necesita así para poder hacer drop() internamente)
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

factsheet = Factsheet(general={"target_column": "target"}, fairness={"protected_feature": "sex","protected_values": [1],"favorable_outcomes": [1]})

# Asumiendo que TrustEvaluator está disponible en tu entorno
evaluator = TrustEvaluator(
    model=wrapped_model,          # <- Pasamos el modelo envuelto
    train_data=train_data,
    test_data=test_data,
    factsheet=factsheet,
    config_path="./src/trust_library/configs.json"
)

subset = evaluator.evaluate(show_nan=True)
print(json.dumps(subset, indent=4))

Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.00 seconds.
Metric 'class_imbalance' computed in 0.00 seconds.
Metric 'kl_divergence' computed in 0.00 se

In [14]:
evaluator.plot_results()

In [15]:
evaluator.plot_radar()

In [17]:
factsheet = Factsheet(general={"target_column": "Target"}, fairness={"protected_feature": "Group","protected_values": [1],"favorable_outcomes": [1]}, save_path="./factsheet_actual.json")

with open("./models_and_data/model_SVM.pkl", "rb") as f:
    loaded_model2 = pickle.load(f)
evaluator2 = TrustEvaluator(loaded_model2, train_loaded, test_loaded, factsheet)
subset2 = evaluator2.evaluate(["fairness", "privacy", "robustness", "accountability", "sustainability"], show_nan=True)



Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.02 seconds.
Metric 'class_imbalance' computed in 0.00 seconds.
Metric 'kl_divergence' computed in 0.00 se

In [18]:

TrustEvaluator.compare_radar({
    "RandomForest": subset,
    "SVM": subset2
})

TrustEvaluator.compare_all_bars({
    "RandomForest": subset,
    "SVM": subset2
})

In [19]:
import numpy as np


def calculate_score(value: float, thresholds: list[float]) -> int:
    """
    Compute a score from 1 to 5 based on the given value and thresholds.
    Detects whether the thresholds are for ascending (accuracy) or descending 
    (error) metrics and calculates the score accordingly.
    """
    if not thresholds or len(thresholds) == 0:
        raise ValueError("Thresholds must be a non empty list.")
        
    value = abs(value) # Trabajamos siempre con valor absoluto
    
    # Caso: Error (Menor es mejor). Ej: [0.075, 0.05, 0.01, 0]
    # Caso: Accuracy (Mayor es mejor). Ej: [0.8, 0.9, 0.95,0.99]
    idx = np.digitize(value, thresholds, right=False)
    score = idx + 1
        
    # Asegurar que el score nunca exceda 5 ni baje de 1
    return int(np.clip(score, 1, 5))

#(1];(2],(3],(4];(5]
calculate_score(0.075, [0.075, 0.05, 0.01, 0])

1

In [21]:
import pandas as pd
import json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Cargar datos
url_train = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
url_test  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

cols = [
    "age","workclass","fnlwgt","education","education-num","marital-status",
    "occupation","relationship","race","sex","capital-gain","capital-loss",
    "hours-per-week","native-country","income"
]

train = pd.read_csv(url_train, names=cols, sep=", ", engine="python", na_values="?")
test  = pd.read_csv(url_test,  names=cols, sep=", ", engine="python", skiprows=1, na_values="?")

train = train.dropna()
test  = test.dropna()

# 2. Mapeo Manual: Convertimos la variable objetivo y la sensible a números controlados
train["income"] = (train["income"] == ">50K").astype(int)
test["income"]  = (test["income"]  == ">50K.").astype(int)

# Convertimos explícitamente a la mujer en 1 y al hombre en 0
train["sex"] = train["sex"].map({"Female": 1, "Male": 0})
test["sex"] = test["sex"].map({"Female": 1, "Male": 0})

# 3. Preprocesamiento general: Convertimos el resto del texto a números
# Usamos OrdinalEncoder en lugar de OneHot para NO borrar la columna 'native-country'
categorical = train.select_dtypes(include=["object"]).columns.tolist()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[categorical] = encoder.fit_transform(train[categorical])
test[categorical] = encoder.transform(test[categorical])

# 4. Separar variables
target = "income"
sensitive = "sex"

# X e y (ahora X SI incluye la columna 'sex', el pipeline se encargará de ella)
X_train = train.drop(columns=[target])
y_train = train[target]
X_test = test.drop(columns=[target])
y_test = test[target]

# 5. Crear el Pipeline "filtro" y entrenar
# Solo pasamos (passthrough) las columnas que NO son el atributo sensible
features_to_keep = X_train.columns.drop(sensitive).tolist()

drop_sensitive = ColumnTransformer([
    ("keep", "passthrough", features_to_keep)
], remainder="drop") # "drop" es la clave aquí: ignora todo lo demás (es decir, 'sex')

lr_model_pipeline = Pipeline([
    ("filter", drop_sensitive),
    ("clf", LogisticRegression(max_iter=1000))
])

# Entrenamos el pipeline completo con X_train (que aún tiene 'sex')
lr_model_pipeline.fit(X_train, y_train)

# 6. Configurar el Factsheet
factsheet2 = Factsheet() # Asumo que tienes esta función definida en tu entorno
factsheet2["general"]["target_column"]["value"] = "income"
factsheet2["fairness"]["protected_feature"]["value"] = "sex"
# OJO AQUÍ: Cambiamos "Female" por 1 porque ya está numerizado
factsheet2["fairness"]["protected_values"]["value"] = [1] 
factsheet2["fairness"]["favorable_outcomes"]["value"] = [1]

factsheet2["privacy"]["quasi_identifiers"]["value"] = ["age", "native-country"]
factsheet2["privacy"]["sensitive_attribute"]["value"] = ["sex"]

# 7. Evaluar (Le pasamos el PIPELINE al evaluador)
evaluator = TrustEvaluator(
    model=lr_model_pipeline, # <--- Pasamos el pipeline, no solo la logística
    train_data=train, 
    test_data=test,   
    factsheet=factsheet2
)

subset = evaluator.evaluate(show_nan=True)
# print(json.dumps(subset, indent=4))
# evaluator.plot_results()

Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.20 seconds.
Metric 'class_imbalance' computed in 0.00 seconds.
Metric 'kl_divergence' computed in 0.00 se

In [22]:
# 1. Definir X e y incluyendo explícitamente el atributo sensible (sex)
X_train_biased = train.drop(columns=["income"])
y_train = train["income"]

# 2. Entrenar el modelo directamente SIN Pipeline
# (Como train ya es 100% numérico, la regresión lo aceptará sin problemas)
biased_model = LogisticRegression(max_iter=1000)
biased_model.fit(X_train_biased, y_train)

# 3. Evaluar el modelo sesgado
evaluator_biased = TrustEvaluator(
    model=biased_model,
    train_data=train,
    test_data=test,
    factsheet=factsheet2
)

subset_biased = evaluator_biased.evaluate(show_nan=True)

# Imprimimos subset formateado
# print(json.dumps(subset_biased, indent=4))
# evaluator_biased.plot_results()

Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.19 seconds.
Metric 'class_imbalance' computed in 0.00 seconds.
Metric 'kl_divergence' computed in 0.00 se

In [23]:
TrustEvaluator.compare_radar({
    "Normal": subset,
    "Biased": subset_biased
})

TrustEvaluator.compare_all_bars({
    "Normal": subset,
    "Biased": subset_biased
})

In [24]:
with open("./models_and_data/model_tree.pkl", "rb") as f:
    loaded_model = pickle.load(f)

train_loaded = pd.read_csv("./models_and_data/train.csv")
test_loaded = pd.read_csv("./models_and_data/test.csv")

evaluator = TrustEvaluator(
    model=loaded_model,
    train_data=train_loaded,
    test_data=test_loaded,
    factsheet=factsheet
)

subset = evaluator.evaluate(show_nan=True)
# Imprimimos subset formateado
print(json.dumps(subset, indent=4))

evaluator.plot_results()

Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.01 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.02 seconds.
Metric 'class_imbalance' computed in 0.00 seconds.
Metric 'kl_divergence' computed in 0.00 se